# Data Skew: Detection, Diagnostics, and Production Remediation

**Module**: 05_partitions_and_shuffle  
**Notebook**: 06_data_skew_and_solutions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Detecting data skew via task duration variance, salting skewed join keys, and enabling AQE skew join optimization.
- **Why does this topic exist?**: Data skew is the single most common cause of straggler tasks and OOM crashes in distributed systems.
- **What problem does it solve?**: 1 task takes 1 hour while 99 tasks take 2 seconds.
- **Where is it used in real-world Data Engineering?**: Joining transactions with massive merchant keys or default customer IDs.

---

## 2. The Data Skew Problem

```
Normal Distribution (Balanced Partitions):
Partition 0: [10,000 rows] -> Task 0 (2s) ===> FINISHED!
Partition 1: [10,000 rows] -> Task 1 (2s) ===> FINISHED!
Partition 2: [10,000 rows] -> Task 2 (2s) ===> FINISHED!

Data Skew (Hotspot on'Null'or Popular Key):
Partition 0: [1,000 rows] -> Task 0 (0.2s) ===> IDLE!
Partition 1: [1,000 rows] -> Task 1 (0.2s) ===> IDLE!
Partition 2: [5,000,000 rows] -> Task 2 (45 mins / OOM Crash!) ===> STRAGGLER!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Skew_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create artificially skewed dataset (90% of rows have key 'HOTSPOT')
skewed_data = [("HOTSPOT", i) for i in range(100000)] + [(f"KEY_{i}", i) for i in range(10000)]
skewed_df = spark.createDataFrame(skewed_data, ["key", "val"])

print("Top keys in skewed dataset:")
skewed_df.groupBy("key").count().orderBy(F.desc("count")).show(3)

---

## Experiment: Remediating Skew via Key Salting

- **1. What are we testing?**: Salt the skewed key with random integers 0-3 to split the hotspot across 4 partitions.
- **2. What do we expect?**: The hotspot key is divided evenly into 4 distinct sub-keys.

In [ ]:
SALT_FACTOR = 4
salted_df = skewed_df.withColumn("salted_key", F.concat(F.col("key"), F.lit("_"), F.floor(F.rand() * SALT_FACTOR)))
print("Salted key distribution:")
salted_df.groupBy("salted_key").count().orderBy(F.desc("count")).show(5)

**3. What happened?**: The 100,000 rows for'HOTSPOT'were split into 4 balanced partitions of ~25,000 rows each.  
**4. Why did it happen?**: Salting distributes identical keys across multiple distinct hash slots.  
**5. Practical Engineering Lesson**: Salting eliminates single-task hotspots on skewed joins.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Skew in Profiling
**Problem Statement**: Profile orders channel column for skew.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Broadcast Join to Avoid Skew
**Problem Statement**: Explain why broadcast join eliminates skew.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Config
**Problem Statement**: Print AQE skew join configuration parameters.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Skew in Profiling):
```python
orders.groupBy('channel').count().orderBy(F.desc('count')).show()
```

#### Solution 2 (Broadcast Join to Avoid Skew):
```python
print('Broadcast join does not shuffle data by key, eliminating key-based skew.')
```

#### Solution 3 (AQE Skew Join Config):
```python
print('skewJoin.enabled:', spark.conf.get('spark.sql.adaptive.skewJoin.enabled','true'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Data Skew**: Uneven key distribution causing straggler tasks.
- **Remediation**: 1. Broadcast Join 2. Salting 3. AQE Skew Join 4. Isolate Skewed Keys.

### Essential Technical Terms
- `Straggler`: Task running significantly slower than peers.
- `Salting`: Appending random integer to split keys.

### Common Engineering Mistakes
- **Adding more executors to fix skew**: Extra executors do not help if 1 task holds all skewed records.

### Interview Questions on This Topic
- **Q: How do you detect data skew in the Spark UI?**
  - *A*: Compare 75th percentile vs Max task duration in the Stages tab; if Max is orders of magnitude larger, skew is present.

### What to Study Next
Module 05 complete! Proceed to Module 06: Memory Architecture and Storage.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')